In [149]:
import pandas
import pathlib
import numpy
import re

In [150]:
files = list(pathlib.Path('.').glob('json/*.json'))

## Collect files

In [151]:
dfs = [pandas.read_json(file) for file in files]

In [152]:
df = pandas.concat(dfs, ignore_index=True)

In [153]:
df.head()

,year,nomination_url,award,nomination_actor,nomination_actor_url,nomination_description,nomination_character_name,nomination_citation,is_winner,film_title,film_url,academy_note,has_acceptance_speech,acceptance_speech_text,acceptance_speech_url
0,1927/28 (1st),https://awardsdatabase.oscars.org/Search/Nomin...,ACTOR,Richard Barthelmess,https://awardsdatabase.oscars.org/Search/Nomin...,NaN,"{""Nickie Elkins""};",None,False,The Noose,https://awardsdatabase.oscars.org/Search/Nomin...,None,False,NaN,NaN
1,1927/28 (1st),https://awardsdatabase.oscars.org/Search/Nomin...,ACTOR,Emil Jannings,https://awardsdatabase.oscars.org/Search/Nomin...,NaN,"{""General Dolgorucki [Grand Duke Sergius Alexa...",None,True,The Last Command,https://awardsdatabase.oscars.org/Search/Nomin...,None,False,NaN,NaN
2,1927/28 (1st),https://awardsdatabase.oscars.org/Search/Nomin...,ACTRESS,Louise Dresser,https://awardsdatabase.oscars.org/Search/Nomin...,NaN,"{""Mrs. Pleznik""}",None,False,A Ship Comes In,https://awardsdatabase.oscars.org/Search/Nomin...,None,False,NaN,NaN
3,1927/28 (1st),https://awardsdatabase.oscars.org/Search/Nomin...,ACTRESS,Janet Gaynor,https://awardsdatabase.oscars.org/Search/Nomin...,NaN,"{""Diane""};",None,True,7th Heaven,https://awardsdatabase.oscars.org/Search/Nomin...,None,False,NaN,NaN
4,1927/28 (1st),https://awardsdatabase.oscars.org/Search/Nomin...,ACTRESS,Gloria Swanson,https://awardsdatabase.oscars.org/Search/Nomin...,NaN,"{""Sadie Thompson""}",None,False,Sadie Thompson,https://awardsdatabase.oscars.org/Search/Nomin...,None,False,NaN,NaN


## Refactoring

In [154]:
df.award = df.award.map(lambda x: isinstance(x, str) and x.lower().title() or x)

The year comes like __2000 (xth edition)__ and we need to correct that

In [155]:
for item in df.itertuples():
    value = df.loc[item.Index, 'year']
    if value is not None:
        year, edition = value.split(' (')
        edition = edition.removesuffix(')').replace('th', '').replace('st', '').replace('nd', '').replace('rd', '')
        df.loc[item.Index, 'year'] = str(year)
        df.loc[item.Index, 'edition'] = int(edition)

In [156]:
def fix_character(value: str | None) -> str | None:
    if value is None:
        return value
    return value.replace('{', '').replace('}', '').replace('"', '').strip()

In [157]:
df.nomination_character_name = df.nomination_character_name.map(fix_character)

In [158]:
def fix_year(value: str | None) -> str | None:
    if value is None:
        return value
    
    if '/' in value:
        parts = value.split('/')
        year = int(parts[0]) + 1
        return year
    return int(value)

In [159]:
df.year = df.year.map(fix_year)

In [160]:
df = df.astype({'year': numpy.int32, 'edition': numpy.int16})

## Foreign Language Film and Song Awargs Fix

Foreign language film provides a country in the actor section. Create a new column `nomination_country` and also a new column called `nomination_producers`

In [161]:
international_awards = df[df.award.str.contains('International', na=False)]

In [162]:
for item in international_awards.itertuples():
    df.loc[item.Index, 'nomination_country'] = item.nomination_actor
    df.loc[item.Index, 'nomination_actor'] = None

In [163]:
song_awards = df[df.award.str.contains('Song', na=False) | df.award.str.contains('Music', na=False)]

In [164]:
for item in song_awards.itertuples():
    df.loc[item.Index, 'nomination_producers'] = item.nomination_actor
    df.loc[item.Index, 'nomination_actor'] = None

## Acceptance speech url

Extract the url to the acceptance speech

In [165]:
def acceptance_speech_url(value: str | None) -> str | None:
    if value is None:
        return value
    
    if isinstance(value, float):
        return value
    
    tokens = value.split(',')
    first_token = tokens[0].replace(('window.open('), '').replace("'", '').strip()
    return first_token

In [166]:
df.acceptance_speech_url = df.acceptance_speech_url.map(acceptance_speech_url)

## Output

In [167]:
df.sort_values(by=['edition', 'award'], ascending=[False, True], inplace=True)

In [168]:
df = df[[
    'year', 
    'edition', 
    'award', ''
    'nomination_actor',
    'nomination_country',
    'nomination_character_name', 
    'nomination_citation',
    'nomination_producers',
    'nomination_description',
    'film_title', 
    'is_winner', 
    'acceptance_speech_text', 
    'acceptance_speech_url'
]]

In [169]:
df.to_csv('oscars.csv', index=False)